# 07 — Gates as Rotations of the Bloch Sphere

See what unitarity buys: a gate is fixed only up to a global phase, it can always be undone, and on the Bloch sphere it is a rigid rotation, never a mirror image.

In `01/06` you showed that a gate is a matrix $U$ with $U^\dagger U = I$: it keeps every overlap, hence every probability. What the rule *gives* you was left for today. You will collect three consequences, and they turn out to be one picture: a gate turns the Bloch sphere.

**Prerequisites:** `01/06_gates_as_unitary_evolution` (the gate $U_\theta$, the dagger and its reversal rule, $U^\dagger U = I$, the shear $M$, the conjugation $K$), `01/05_inner_product_and_dirac` (§5 overlap from the angle between Bloch arrows, §6 global phase), `01/03_the_bloch_sphere` (the coordinates $\theta, \phi$ and the Bloch vector).

**What you'll be able to do**
- Show that $U$ and $e^{i\alpha}U$ are the same physical gate, and tell a global phase on a gate from a phase on one column, which makes a different gate.
- Undo any gate with $U^{-1} = U^\dagger$, undo a sequence of gates in the reverse order, and explain why a measurement cannot be undone.
- Compute by hand where $U_\theta$ sends $|0\rangle$ on the Bloch sphere, and see the half-angle in the matrix become a full angle on the sphere.
- Build, column by column, the $3\times3$ matrix by which a gate acts on Bloch vectors, check that it is a rotation (angles kept, determinant $+1$) on $200$ random gates, and meet the mirror image that no gate can produce.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, bloch_vector
from qot_course.quantum.gates import apply_gate, is_unitary

rng = np.random.default_rng(0)   # random gates and random states in sections 2 and 4
viz.use_course_style()

def U_theta(t):
    '''The unnamed gate of 01/06: [[cos t/2, -sin t/2], [sin t/2, cos t/2]] (01/10 will name it).'''
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]], dtype=complex)

def random_unitary(rng):
    '''A random 2x2 unitary: QR makes the columns of a random complex matrix orthonormal (01/06 §3);
    multiplying each column by a phase keeps them orthonormal.'''
    # QR is a library routine that orthonormalises columns; we use it only as a generator
    z = rng.normal(size=(2, 2)) + 1j * rng.normal(size=(2, 2))
    q, r = np.linalg.qr(z)
    return q * (np.diag(r) / np.abs(np.diag(r)))

theta = 2 * np.pi / 3
U = U_theta(theta)                 # the running example of 01/06
U_dag = U.conj().T

## 1. Two matrices, one gate: the global phase

`01/02` and `01/05` established that a global phase $e^{i\alpha}$ on a state is unobservable: $|\psi\rangle$ and $e^{i\alpha}|\psi\rangle$ have the same Born shadows and the same Bloch point, and their overlap with any $|\varphi\rangle$ differs only by the factor $e^{i\alpha}$, which has modulus $1$. The same is true of gates: $U$ and $e^{i\alpha}U$ are *different matrices* but the *same physical gate*, since $e^{i\alpha}U|\psi\rangle$ is $U|\psi\rangle$ times a global phase, for every $|\psi\rangle$. And $e^{i\alpha}U$ is still unitary — check: $(e^{i\alpha}U)^\dagger(e^{i\alpha}U) = e^{-i\alpha}e^{i\alpha}\,U^\dagger U = I$, because the dagger conjugates the scalar $e^{i\alpha}$ into $e^{-i\alpha}$. So "valid gate = unitary matrix" is exact, with the understanding that unitary matrices differing by a global phase are the same gate. Keep this in mind for `01/10`, where two named gates will turn out to be the same rotation up to such a phase.

**A phase on one column is a different gate.** The word *global* matters. Compare $U$ with $U\,\mathrm{diag}(1, e^{i\alpha})$, which multiplies only the second column of $U$ by $e^{i\alpha}$. On $|0\rangle$ nothing changes, because the first column is untouched. On $|+\rangle$, linearity gives $\tfrac{1}{\sqrt2}\big(U|0\rangle + e^{i\alpha}\,U|1\rangle\big)$: the phase now sits between the two parts, a *relative* phase, which `01/02` showed is visible. By hand at $\alpha = \pi$, the second column changes sign, so the new gate sends $|+\rangle$ to $\tfrac{1}{\sqrt2}(U|0\rangle - U|1\rangle) = U|-\rangle$. Since $|+\rangle$ and $|-\rangle$ are orthogonal and $U$ keeps overlaps, $U|+\rangle$ and $U|-\rangle$ are orthogonal too: the two gates send the same input to two perfectly distinguishable outputs. With numbers, $U|+\rangle = (-0.259, 0.966)$ and $U|-\rangle = \tfrac{1}{\sqrt2}(0.5 + 0.866,\ 0.866 - 0.5) = (0.966, 0.259)$, whose overlap is $-0.25 + 0.25 = 0$.

Let's check both statements, then sweep $\alpha$ from $0$ to $2\pi$ and watch the Bloch coordinates of the output under each kind of phase.

In [ ]:
alpha = 0.7
U_phase = np.exp(1j * alpha) * U
print("e^{i alpha} U is unitary?         ", is_unitary(U_phase))
print("Bloch vector of U|0>              :", np.round(bloch_vector(apply_gate(U, KET_0)), 6) + 0)
print("Bloch vector of e^{i alpha} U|0>  :", np.round(bloch_vector(apply_gate(U_phase, KET_0)), 6) + 0, " (same point)")
print("the amplitudes themselves differ  :", np.round(apply_gate(U_phase, KET_0), 3), " vs ", np.round(apply_gate(U, KET_0), 3))
psi_r = qubit_state(1.1, 2.3)    # an arbitrary tilted state, off every axis
print("on a tilted state, same point?    :", np.allclose(bloch_vector(apply_gate(U_phase, psi_r)), bloch_vector(apply_gate(U, psi_r))))

# A phase on ONE column: a different gate
col_phase = lambda a: U @ np.diag([1, np.exp(1j * a)])
print("\nU|+>                      =", np.round(apply_gate(U, KET_PLUS), 3))
print("U diag(1,-1)|+>  (= U|->)  =", np.round(apply_gate(col_phase(np.pi), KET_PLUS), 3) + 0)
print("overlap of the two outputs =", np.round(np.vdot(apply_gate(U, KET_PLUS), apply_gate(col_phase(np.pi), KET_PLUS)), 6) + 0)
print("Bloch vectors              :", np.round(bloch_vector(apply_gate(U, KET_PLUS)), 3) + 0, " vs ",
      np.round(bloch_vector(apply_gate(col_phase(np.pi), KET_PLUS)), 3) + 0)

print("column phase at alpha = pi/2, 3pi/2:", np.round(bloch_vector(apply_gate(col_phase(np.pi / 2), KET_PLUS)), 3) + 0,
      np.round(bloch_vector(apply_gate(col_phase(3 * np.pi / 2), KET_PLUS)), 3) + 0)

alphas = np.linspace(0, 2 * np.pi, 181)
glob = np.array([bloch_vector(apply_gate(np.exp(1j * a) * U, KET_PLUS)) for a in alphas])
amp0 = np.array([apply_gate(np.exp(1j * a) * U, KET_PLUS)[0].real for a in alphas])
rel = np.array([bloch_vector(apply_gate(col_phase(a), KET_PLUS)) for a in alphas])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), sharey=True, gridspec_kw={"wspace": 0.12})
for ax, data, title in [(axes[0], glob, r"Global phase: $e^{i\alpha}U$ on $|+\rangle$"),
                        (axes[1], rel, r"Phase on one column: $U\,\mathrm{diag}(1, e^{i\alpha})$ on $|+\rangle$")]:
    for k, (lbl, col) in enumerate([("x", COLORS["source"]), ("y", COLORS["flow"]), ("z", COLORS["target"])]):
        ax.plot(alphas / np.pi, data[:, k], color=col, lw=2.4, label=f"Bloch {lbl}")
    ax.set_xlabel(r"$\alpha/\pi$"); ax.set_title(title, fontsize=12); ax.set_ylim(-1.1, 1.1)
axes[0].plot(alphas / np.pi, amp0, color=COLORS["muted"], ls=":", lw=2, label=r"Re of the amplitude $a_0$")
axes[0].set_ylabel("value"); axes[0].legend(loc="lower left", fontsize=9)
plt.show()

**Read the output and the figure.** The amplitudes of $e^{i\alpha}U|0\rangle$ differ from those of $U|0\rangle$ (a common factor $e^{0.7i}$), yet the Bloch point is identical, on $|0\rangle$ and on a tilted state alike. The phase on one column does something else: at $\alpha = \pi$ it sends $|+\rangle$ to $U|-\rangle = (0.966, 0.259)$, orthogonal to $U|+\rangle$ (overlap $0$), and the two Bloch vectors $(-0.5, 0, -0.866)$ and $(0.5, 0, 0.866)$ point in opposite directions.

In the figure, the left panel follows the output of $e^{i\alpha}U$ on $|+\rangle$ as $\alpha$ makes a full turn. The dotted grey curve, the real part of the first amplitude, swings up and down: the matrix really changes. The three Bloch coordinates stay perfectly flat, at $x = -0.5$, $y = 0$, $z = -0.866$: nothing measurable changes. On the right, the same sweep with the phase on one column moves the output around the sphere. $x$ and $z$ both follow $\cos\alpha$: they cross zero together at $\alpha = \pi/2$ and reach $(0.5, 0, 0.866)$ at $\alpha = \pi$, the antipodal point found above, while $y = \sin\alpha$ swings up to $+1$ and down to $-1$ in between. A global phase is invisible; the same phase on one column is a different gate.

## 2. Every gate can be undone

The **inverse** of a matrix $U$ is the matrix that undoes it: $U^{-1}U = I$. For a unitary $U$ we already hold it in our hands — $U^\dagger U = I$ says precisely that

$$ U^{-1} = U^\dagger . $$

So every gate can be *undone* by another gate (its dagger, itself unitary): quantum evolution of a closed system is reversible, and no information is lost along the way.

**Undoing a sequence.** Apply $U_1$, then $U_2$, then $U_3$: the overall gate is the product $U_3U_2U_1$ (rightmost first, because $U_1$ meets the ket first: `01/06` §1). The reversal rule of `01/06` §2 gives its dagger at once,

$$ (U_3U_2U_1)^\dagger = U_1^\dagger\,U_2^\dagger\,U_3^\dagger , $$

so to undo the sequence you undo the *last* gate first, as you take off your shoes before your socks. Undoing in the original order, $U_3^\dagger U_2^\dagger U_1^\dagger$, does not return the state in general, because gates do not commute.

**What cannot be undone.** A measurement that reports outcome $0$ leaves the qubit in $|0\rangle$, whether the input was $|0\rangle$ or $|+\rangle$. Two different inputs, with overlap $\tfrac{1}{\sqrt2} \approx 0.707$, end as one output, with overlap $1$. No later operation, gate or otherwise, can tell which input came in, so the measurement cannot be undone. It is not a gate (`01/06` exercise 2 asks you to prove it from the overlaps), and the general non-unitary maps are the subject of `01/20`. An invertible matrix is not enough either: the shear $M$ of `01/06` has the inverse $M^{-1} = \begin{pmatrix}1 & -1\\ 0 & 1\end{pmatrix}$, but neither of the two is unitary. A gate is reversible *by another gate*.

Let's undo $U$, then send a state out through three random gates and bring it back, once in the right order and once in the wrong one, following its overlap with the starting state at each step.

In [ ]:
out = apply_gate(U, KET_0)
recovered = apply_gate(U_dag, out)     # apply U-dagger to undo U
print("U|0>                    :", np.round(out, 6))
print("U-dagger applied after U:", np.round(recovered, 6) + 0)
print("back to |0>?            ", np.allclose(recovered, KET_0))
print("U-dagger is unitary too?", is_unitary(U_dag))

# Out through three random gates, then back: right order vs wrong order
gates = [random_unitary(rng) for _ in range(3)]
print("the three random gates are unitary:", [is_unitary(G) for G in gates])
psi0 = qubit_state(1.1, 2.3)
fid = lambda v: abs(np.vdot(psi0, v)) ** 2       # squared overlap with the starting state

def journey(back_order):
    v, path = psi0.copy(), [1.0]
    for G in gates:                               # out: U1, then U2, then U3
        v = G @ v; path.append(fid(v))
    for G in back_order:                          # back: daggers in the given order
        v = G.conj().T @ v; path.append(fid(v))
    return np.array(path)

right = journey(gates[::-1])      # undo U3 first, then U2, then U1
wrong = journey(gates)            # undo U1 first: the wrong order
print("squared overlap with the start, right order:", np.round(right, 3))
print("squared overlap with the start, wrong order:", np.round(wrong, 3))

# Measurement: two inputs, one output
meas0 = lambda v: np.array([1.0, 0.0], dtype=complex)    # outcome 0 leaves |0>, whatever came in
print(f"\n|<0|+>| before = {abs(np.vdot(KET_0, KET_PLUS)):.3f}   after the measurement = {abs(np.vdot(meas0(KET_0), meas0(KET_PLUS))):.3f}")
M = np.array([[1.0, 1.0], [0.0, 1.0]], dtype=complex)
print("shear M: inverse =", np.linalg.inv(M).real.tolist(), "  M unitary?", is_unitary(M), "  M^-1 unitary?", is_unitary(np.linalg.inv(M)))

fig, ax = plt.subplots(figsize=(10.5, 4.2))
steps = np.arange(7)
ax.plot(steps, right, "o-", color=COLORS["quantum"], lw=2.4, ms=8, label=r"back in reverse order: $U_1^\dagger U_2^\dagger U_3^\dagger$")
ax.plot(steps, wrong, "s--", color=COLORS["negative"], lw=2, ms=7, label=r"back in the original order: $U_3^\dagger U_2^\dagger U_1^\dagger$")
ax.axvspan(-0.3, 3, color=COLORS["grid"], alpha=0.35)
ax.text(1.5, 1.08, "out: $U_1$, $U_2$, $U_3$", ha="center"); ax.text(4.5, 1.08, "back: three daggers", ha="center")
ax.set_xticks(steps); ax.set_xticklabels(["start", "$U_1$", "$U_2$", "$U_3$", "dagger 1", "dagger 2", "dagger 3"])
ax.set_ylim(0, 1.18); ax.set_xlim(-0.3, 6.3)
ax.set_ylabel(r"$|\langle\psi_0|\psi_k\rangle|^2$"); ax.set_title("Out and back through three random gates")
ax.legend(loc="lower left", fontsize=9)
plt.show()

**Read the output and the figure.** Applying $U^\dagger$ after $U$ returns the state exactly to $|0\rangle$, and $U^\dagger$ is itself a legitimate gate. Out and back with nothing lost — the hallmark of unitary evolution. The measurement does the opposite: two inputs with overlap $0.707$ leave with overlap $1.000$, merged for good. The shear has an inverse matrix, but neither it nor its inverse passes `is_unitary`.

In the figure, the shaded part is the way out. The starting state wanders away under the three random gates (its squared overlap with the start drops step by step, down to $0.287$ with this seed). On the way back, the blue curve retraces the same values in mirror order and lands exactly on $1$ at the last step: undoing $U_3$ first, then $U_2$, then $U_1$ returns the state. The dashed coral curve undoes the gates in the original order and ends below $1$, at $0.726$ with this seed: the right daggers in the wrong order do not return the state in general. Reversibility comes with an order, the one the dagger's reversal rule dictates.

## 3. A gate turns the Bloch sphere

Now the geometric payoff. In `01/05` §5 you saw (and its exercise 3 asks you to prove) that the overlap of two pure states is a function of the angle $\Theta$ between their Bloch arrows: $|\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r_\varphi\cdot\vec r_\psi) = \cos^2(\Theta/2)$. A unitary preserves every overlap (`01/06` §3), so it preserves the angle between *any* two Bloch arrows: it moves the whole sphere **rigidly**, like turning a globe in your hands. Section 4 shows that the motion is always a rotation; here we watch one gate do it, and there is a factor of two to watch.

**By hand.** $U|0\rangle = (\cos\tfrac{\theta}{2},\ \sin\tfrac{\theta}{2})$ has real, non-negative amplitudes, so in the coordinates of `01/03` its polar angle is $\theta$ and its azimuth is $0$: Bloch vector $(\sin\theta,\ 0,\ \cos\theta)$. For $\theta = 2\pi/3$ that is $(0.866,\ 0,\ -0.5)$ — the arrow swung from the north pole down through $120^\circ$ in the $x$–$z$ plane. The matrix carries the **half**-angle $\theta/2 = 60^\circ$; the sphere shows the **full** angle $\theta = 120^\circ$. That is the same factor of two `01/03` built into the sphere ($\cos\tfrac{\theta}{2}$ in the state, $\cos\theta$ in the coordinate) — now it shows up as "half-angle in the gate, full angle in the rotation".

The factor of two has a striking consequence. At $\theta = 2\pi$ the arrow has made a full turn and is back at the north pole, yet the matrix is $U_{2\pi} = \begin{pmatrix}\cos\pi & -\sin\pi\\ \sin\pi & \cos\pi\end{pmatrix} = -I$, not $I$. Section 1 removes the paradox: $-I = e^{i\pi}I$ differs from $I$ by a global phase, so it is the same gate, the one that does nothing. The matrix returns to $I$ itself only at $\theta = 4\pi$.

Let's confirm the numbers, then sweep $\theta$ from $0$ to $2\pi$ to watch $U_\theta|0\rangle$ trace its path, and plot the angle on the sphere against the angle written in the matrix.

In [ ]:
r_in, r_out = bloch_vector(KET_0), bloch_vector(out)
print("Bloch vector of |0>          :", np.round(r_in, 3) + 0, " (north pole)")
print("Bloch vector of U|0> (numpy) :", np.round(r_out, 3) + 0)
print("by hand (sin th, 0, cos th)  :", np.round([np.sin(theta), 0.0, np.cos(theta)], 3) + 0)
print("angle between the two arrows :", round(np.degrees(np.arccos(np.clip(r_in @ r_out, -1, 1))), 1), "deg  (= theta, the FULL angle)")
print("U_{2 pi} =\n", np.round(U_theta(2 * np.pi), 6).real + 0, "\nU_{4 pi} =\n", np.round(U_theta(4 * np.pi), 6).real + 0)

# Sweep: where does U_theta send |0> as theta runs from 0 to 2*pi?
thetas = np.linspace(0, 2 * np.pi, 13)
path = np.array([bloch_vector(apply_gate(U_theta(t), KET_0)) for t in thetas])
fine = np.linspace(0, 2 * np.pi, 361)
sphere_angle = np.array([np.degrees(np.arccos(np.clip(r_in @ bloch_vector(apply_gate(U_theta(t), KET_0)), -1, 1))) for t in fine])
print("radius of every swept point  :", np.round(np.linalg.norm(path, axis=1), 6))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.3})
ax = axes[0]
circle = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.sin(circle), np.cos(circle), color=COLORS["grid"], lw=1.5, label="Bloch sphere, x–z slice")
ax.plot(path[:, 0], path[:, 2], "o", color=COLORS["quantum"], ms=6, label=r"$U_\theta|0\rangle$, $\theta = 0, \pi/6, \dots, 2\pi$")
ax.plot([0, r_in[0]], [0, r_in[2]], color=COLORS["source"], lw=2.5, label=r"$|0\rangle$ (input)")
ax.plot([0, r_out[0]], [0, r_out[2]], color=COLORS["highlight"], lw=2.5, label=r"$U|0\rangle$, $\theta = 2\pi/3$")
for lbl, xy in [(r"$|0\rangle$", (0, 1.13)), (r"$|1\rangle$", (0, -1.15)), (r"$|+\rangle$", (1.17, 0)), (r"$|-\rangle$", (-1.2, 0))]:
    ax.text(*xy, lbl, ha="center", va="center")
ax.set_aspect("equal"); ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35)
ax.set_xlabel("x"); ax.set_ylabel("z"); ax.set_title(r"Sweeping $\theta$: $|0\rangle$ turns about the $y$ axis", fontsize=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=8, frameon=False)
ax = axes[1]
ax.plot(np.degrees(fine), sphere_angle, color=COLORS["quantum"], lw=2.4, label=r"angle on the sphere between $|0\rangle$ and $U_\theta|0\rangle$")
ax.plot(np.degrees(fine), np.degrees(fine) / 2, color=COLORS["target"], ls="--", lw=2, label=r"angle written in the matrix, $\theta/2$")
ax.plot(np.degrees(fine), np.degrees(fine), color=COLORS["muted"], ls=":", lw=1.5, label=r"the full angle $\theta$")
ax.plot(120, 120, "o", color=COLORS["highlight"], ms=9)
ax.set_xlabel(r"$\theta$ (degrees)"); ax.set_ylabel("angle (degrees)"); ax.set_xticks(range(0, 361, 60)); ax.set_yticks(range(0, 361, 60))
ax.set_ylim(0, 370); ax.set_title("Half-angle in the matrix, full angle on the sphere", fontsize=12); ax.legend(loc="upper left", fontsize=9)
plt.show()

**Read the output and the figure.** The numbers agree: NumPy's Bloch vector of $U|0\rangle$ is $(0.866, 0, -0.5)$, exactly the by-hand $(\sin\theta, 0, \cos\theta)$, and the angle between input and output arrows is $120^\circ = \theta$ — the full angle, from a matrix written with $\theta/2$. $U_{2\pi}$ prints as $-I$ and $U_{4\pi}$ as $I$, as computed by hand.

On the left, the twelve distinct dots of the sweep (the thirteenth, at $\theta = 2\pi$, falls back on the first) sit on the unit circle of the $x$–$z$ slice, evenly spaced by $30^\circ$, and every one has radius $1$. As $\theta$ grows, $U_\theta$ carries $|0\rangle$ down through $|+\rangle$ (at $\theta = \pi/2$), to $|1\rangle$ (at $\theta = \pi$), on through $|-\rangle$, and back to $|0\rangle$ at $\theta = 2\pi$ — a rotation about the axis perpendicular to the page, the $y$ axis (with $x$ to the right and $z$ up, it points into the page). Every dot stays on the sphere, exactly what unitarity guarantees. (`01/10` will name this family $R_y(\theta)$; today it is enough to see it turn.)

On the right, the blue curve is the angle you would measure on the sphere. Up to $\theta = 180^\circ$ it lies exactly on the dotted line of the full angle $\theta$, not on the dashed line of the half-angle $\theta/2$ that appears in the matrix; the pink dot marks our $120^\circ$. Beyond $180^\circ$ it comes back down, because the arrow passes the south pole and approaches the north pole again from the other side, so the angle *between* the two arrows shrinks to $0$ at $\theta = 360^\circ$, where the matrix is $-I$.

## 4. A rotation, never a mirror image

A rigid motion of the sphere about its centre is a $3\times3$ real matrix $R$ acting on Bloch vectors, $\vec r \mapsto R\,\vec r$, with orthonormal columns, $R^{\mathsf T}R = I$ — the real cousin of the column test of `01/06` §3. The **determinant** of a square matrix is the signed volume of the box spanned by its columns: for $2\times2$, $\det\begin{pmatrix}a&b\\c&d\end{pmatrix} = ad - bc$; for $3\times3$, expand along any row, each entry times the $2\times2$ determinant left when its row and column are struck out, with signs alternating $+,-,+$. Orthonormal columns span a unit cube, so $\det R = \pm1$, and the sign says whether the box kept or reversed its handedness. One more fact, taken on trust: $\det(AB) = \det A\,\det B$. There are two kinds. If $\det R = +1$, $R$ is a **rotation** about some axis. If $\det R = -1$, $R$ is a rotation combined with a **mirror reflection**, and turns a right hand into a left one. Keeping angles cannot tell the two kinds apart; the determinant can.

**Building $R$ by hand.** The columns of $R$ are the images of the three axis arrows — the column rule of `01/06` §1, now on the sphere. The arrow $(1, 0, 0)$ belongs to $|+\rangle$, the arrow $(0, 1, 0)$ to $|{+i}\rangle = \tfrac{1}{\sqrt2}(|0\rangle + i|1\rangle)$, and $(0, 0, 1)$ to $|0\rangle$. So the first column of $R$ is the Bloch vector of $U|+\rangle$, the second that of $U|{+i}\rangle$, the third that of $U|0\rangle$. For $U_\theta$, write $c = \cos\tfrac{\theta}{2}$ and $s = \sin\tfrac{\theta}{2}$:
- $U|0\rangle = (c, s)$, Bloch vector $(\sin\theta, 0, \cos\theta)$ (section 3).
- $U|+\rangle = \tfrac{1}{\sqrt2}(c - s,\ s + c)$, real amplitudes, so $y = 0$; with the amplitude formulas of `01/03`, $x = (c - s)(c + s) = c^2 - s^2 = \cos\theta$ and $z = \tfrac12\big((c - s)^2 - (c + s)^2\big) = -2cs = -\sin\theta$.
- $U|{+i}\rangle = \tfrac{1}{\sqrt2}(c - is,\ s + ic) = \tfrac{1}{\sqrt2}\,e^{-i\theta/2}\,(1,\ i)$, because $s + ic = i(c - is)$. That is $|{+i}\rangle$ times a global phase: the arrow $(0, 1, 0)$ does not move. It is the **axis** of the rotation.

So

$$ R = \begin{pmatrix} \cos\theta & 0 & \sin\theta \\ 0 & 1 & 0 \\ -\sin\theta & 0 & \cos\theta \end{pmatrix}, \qquad \det R = \cos^2\theta + \sin^2\theta = 1 , $$

(expand along the middle row: only the $1$ contributes, times $\cos^2\theta + \sin^2\theta$), the turn by the full angle $\theta$ about the $y$ axis seen in section 3. Two facts are taken on trust here and checked numerically below: that the Bloch map of any gate is linear, so that $R\,\vec r_\psi$ is the Bloch vector of $U|\psi\rangle$ for every state, and that $\det R$ is always $+1$. The reason for the second, in one line: every unitary can be reached from $I$ by turning a dial continuously ($U_\theta$ is one such dial), $\det R$ then changes continuously, and a quantity that can only be $+1$ or $-1$ cannot jump; it starts at $+1$ for $I$. The first step of that sentence, that every unitary is connected to $I$, is itself a theorem we do not prove.

**The mirror that no gate makes.** Recall the conjugation $K(a_0, a_1) = (\bar a_0, \bar a_1)$ of `01/06` §4, which keeps every squared overlap but is not linear. On the amplitude formulas of `01/03`, $x + iy = 2\,\bar a_0 a_1$ becomes $2\,a_0\bar a_1 = x - iy$, and $z = |a_0|^2 - |a_1|^2$ does not change. So $K$ sends $(x, y, z)$ to $(x, -y, z)$: the reflection in the $x$–$z$ plane, with determinant $-1$. Angles kept, handedness reversed. This is the geometric face of Wigner's theorem (`01/06` §3): the unitary maps turn the sphere, the anti-unitary maps reverse its handedness (a mirror, possibly followed by a turn), and the gates, being linear, are on the rotation side. Because $R(e^{i\alpha}U) = R(U)$ (section 1: the sphere does not see a global phase), $U$ and $-U$ give the same rotation, and the $-I$ of section 3 gives the identity rotation.

Let's build $R$ for our $U$, test it on random states, then compute $\det R$ for $200$ random gates and for $K$.

In [ ]:
KET_PLUS_I = np.array([1, 1j], dtype=complex) / np.sqrt(2)

def bloch_map(G):
    '''3x3 matrix of the gate G on Bloch vectors: columns = Bloch vectors of G|+>, G|+i>, G|0>.'''
    return np.column_stack([bloch_vector(apply_gate(G, k)) for k in (KET_PLUS, KET_PLUS_I, KET_0)])

R = bloch_map(U)
R_hand = np.array([[np.cos(theta), 0, np.sin(theta)], [0, 1, 0], [-np.sin(theta), 0, np.cos(theta)]])
print("R for U (numpy) =\n", np.round(R, 3) + 0)
print("matches the hand result:", np.allclose(R, R_hand), "   R^T R = I:", np.allclose(R.T @ R, np.eye(3)), "   det R =", round(np.linalg.det(R), 6))
print("U|+i> = e^{-i theta/2}|+i>?", np.allclose(apply_gate(U, KET_PLUS_I), np.exp(-1j * theta / 2) * KET_PLUS_I))

# Linearity of the Bloch map, tested on random states
# arccos(1 - 2u) spreads the polar angle so that points are uniform on the sphere
states = [qubit_state(np.arccos(1 - 2 * rng.uniform()), 2 * np.pi * rng.uniform()) for _ in range(200)]
print("R r_psi == r_(U psi) on 200 random states:", all(np.allclose(R @ bloch_vector(v), bloch_vector(U @ v)) for v in states))

# 200 random gates: is each Bloch map a rotation?
dets, angle_pairs = [], []
for k in range(200):
    G = random_unitary(rng)
    Rg = bloch_map(G)
    assert is_unitary(G) and np.allclose(Rg.T @ Rg, np.eye(3), atol=1e-9)
    dets.append(np.linalg.det(Rg))
    a, b = states[k], states[(k + 1) % 200]
    angle_pairs.append((np.degrees(np.arccos(np.clip(bloch_vector(a) @ bloch_vector(b), -1, 1))),
                        np.degrees(np.arccos(np.clip(bloch_vector(G @ a) @ bloch_vector(G @ b), -1, 1)))))
dets, angle_pairs = np.array(dets), np.array(angle_pairs)
print(f"\n200 random gates: det R from {dets.min():.6f} to {dets.max():.6f};  largest angle change {np.max(np.abs(angle_pairs[:, 0] - angle_pairs[:, 1])):.1e} deg")
G = random_unitary(rng)
print("global phase forgotten: R(e^{0.7i} G) == R(G):", np.allclose(bloch_map(np.exp(0.7j) * G), bloch_map(G)),
      "   R(-G) == R(G):", np.allclose(bloch_map(-G), bloch_map(G)))

# The conjugation K: a mirror
K_map = np.column_stack([bloch_vector(np.conj(k)) for k in (KET_PLUS, KET_PLUS_I, KET_0)])
print("\nBloch map of K =\n", np.round(K_map, 3) + 0, "\ndet =", round(np.linalg.det(K_map), 6))
k_angles = np.array([(np.degrees(np.arccos(np.clip(bloch_vector(a) @ bloch_vector(b), -1, 1))),
                      np.degrees(np.arccos(np.clip(bloch_vector(np.conj(a)) @ bloch_vector(np.conj(b)), -1, 1))))
                     for a, b in zip(states[:100], states[100:])])
print(f"K keeps angles too: largest change {np.max(np.abs(k_angles[:, 0] - k_angles[:, 1])):.1e} deg")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot([0, 180], [0, 180], color=COLORS["muted"], ls="--", lw=1.2, label="unchanged (diagonal)")
ax.scatter(angle_pairs[:, 0], angle_pairs[:, 1], s=14, color=COLORS["quantum"], label="random gate, random pair")
ax.scatter(k_angles[:, 0], k_angles[:, 1], s=22, facecolors="none", edgecolors=COLORS["negative"], label="conjugation K, random pair")
ax.set_xlabel("angle between two Bloch arrows before (deg)"); ax.set_ylabel("the same angle after (deg)")
ax.set_xlim(0, 180); ax.set_ylim(0, 180); ax.set_title("Angles: gates and K both keep them"); ax.legend(loc="upper left", fontsize=9)
ax = axes[1]
ax.axhline(0, color=COLORS["grid"], lw=1)
ax.scatter(np.arange(200), dets, s=14, color=COLORS["quantum"], label=r"$\det R$ for 200 random gates")
ax.scatter([100], [np.linalg.det(K_map)], s=140, marker="X", color=COLORS["negative"], label=r"$\det$ for the conjugation $K$")
ax.set_ylim(-1.4, 1.4); ax.set_yticks([-1, 0, 1]); ax.set_xlabel("random gate number"); ax.set_ylabel("determinant of the Bloch map")
ax.set_title("Handedness: gates +1, the mirror -1"); ax.legend(loc="center right", fontsize=9)
plt.show()

**Read the output and the figure.** The $3\times3$ matrix built from three Bloch vectors matches the hand result entry by entry, has orthonormal columns and determinant $1$, and $U|{+i}\rangle$ is $|{+i}\rangle$ times $e^{-i\theta/2}$, the fixed axis. For our $U$, on $200$ random states, $R\,\vec r_\psi$ is the Bloch vector of $U|\psi\rangle$: a check, on one gate, of the linearity we took on trust. On $200$ random gates, every Bloch map has orthonormal columns and a determinant equal to $1$ up to rounding, and every pair of arrows keeps its angle. A global phase on a gate, including the sign $-1$, leaves the rotation unchanged. The conjugation gives the matrix $\mathrm{diag}(1, -1, 1)$ with determinant $-1$, and it keeps angles too.

In the figure, the left panel shows why angles alone cannot decide: the blue dots (gates) and the coral rings (conjugation) all sit on the diagonal. Both kinds of map are rigid. The right panel separates them. All $200$ random gates have a determinant on the line at $+1$; the conjugation sits alone at $-1$. Every gate turns the sphere; none of them reflects it. The named gates of the next notebooks are the most useful axes and angles.

## Your turn

1. **Same gate or not?** For each pair, decide by hand whether the two matrices are the same physical gate, then check by comparing the Bloch vectors of their outputs on a tilted state such as `qubit_state(1.1, 2.3)`: (a) $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ and $\begin{pmatrix}0&i\\i&0\end{pmatrix}$; (b) $\mathrm{diag}(1, -1)$ and $\mathrm{diag}(-1, 1)$; (c) $\mathrm{diag}(1, i)$ and $\mathrm{diag}(i, 1)$. (Hint for (c): factor $i$ out of the second matrix and compare what is left with the first.)
2. **Undoing is turning back.** Write $U_\theta^\dagger$ by hand and show that it equals $U_{-\theta}$: the inverse of a turn by $\theta$ is the turn by $-\theta$ about the same axis. Check in code that $U_{-\theta}U_\theta|0\rangle = |0\rangle$, and that the Bloch map of $U_\theta^\dagger$ is the transpose of the Bloch map of $U_\theta$. (For a rotation matrix, $R^{-1} = R^{\mathsf T}$, the real cousin of $U^{-1} = U^\dagger$.)
3. **The rotation of a phase gate.** For $U_\varphi = \mathrm{diag}(1, e^{i\varphi})$ with $\varphi = \pi/2$ (exercise 1 of `01/06`), build the $3\times3$ Bloch map column by column *by hand*: find the Bloch vectors of $U_\varphi|+\rangle$, $U_\varphi|{+i}\rangle$ and $U_\varphi|0\rangle$. Recognise a quarter-turn about the $z$ axis, compute its determinant, and confirm with `bloch_map`.
4. **Harder: rotations compose.** (a) Show numerically, on $100$ random pairs of gates from `random_unitary`, that the Bloch map of a product is the product of the Bloch maps: $R(VU) = R(V)\,R(U)$. (b) Deduce that $\det R(VU) = \det R(V)\det R(U)$, so a circuit of single-qubit gates is always a rotation (the determinant of a product is the product of the determinants; see section 4). (c) Section 4 showed that $e^{i\alpha}U$ and $U$ give the same rotation. Prove the converse: if $R(V) = R(U)$, then $V = e^{i\alpha}U$. Set $W = V^\dagger U$; using (a) and exercise 2, show $R(W) = I$. Deduce that $W$ sends $|0\rangle$ and $|1\rangle$ to themselves up to a phase, so $W = \mathrm{diag}(e^{ia}, e^{ib})$, and that $W|+\rangle \propto |+\rangle$ forces $a = b$. Then explain with section 3 why $U_{2\pi} = -I$ makes the map from matrices of determinant $1$ to rotations two-to-one.

## What you built

- You can show that $U$ and $e^{i\alpha}U$ are the same physical gate, and tell that global phase from a phase on one column, which sends $|+\rangle$ to an orthogonal output when $\alpha = \pi$.
- You can undo any gate with $U^{-1} = U^\dagger$, undo a sequence in the reverse order, and explain why a measurement, which merges different inputs, cannot be undone, and why an invertible matrix such as the shear is still no gate.
- You can compute by hand the Bloch vector of $U_\theta|0\rangle$, see the half-angle in the matrix become a full angle on the sphere, and explain why $U_{2\pi} = -I$ is the gate that does nothing.
- You can build the $3\times3$ Bloch map of a gate column by column, find its axis by hand, and check on $200$ random gates that it is a rotation, with determinant $+1$.
- You can name the mirror that no gate makes: complex conjugation keeps every angle but reverses handedness, the geometric face of Wigner's theorem.

`01/06` told you which matrices are gates; today you saw what they do. Every single-qubit gate turns the Bloch sphere, and that picture will carry you through the named gates of the next three notebooks.

## What's next

Time to name names. In `01/08_pauli_gates` we meet the three most important single-qubit gates — $X$, $Y$, $Z$ — and uncover their double life: they are gates you apply *and* the observables whose averages are the Bloch coordinates. You will read each one off its columns, as in `01/06`, and check that each one is a half-turn of the sphere about its own axis, as today.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, chapter 2 "Introduction to quantum mechanics" and chapter 4 "Quantum circuits", Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- J. J. Sakurai & J. Napolitano, *Modern Quantum Mechanics*, chapter 1 "Fundamental Concepts" and chapter 2 "Quantum Dynamics", 3rd ed., Cambridge University Press (2020). doi:10.1017/9781108587280
- V. Bargmann, "Note on Wigner's theorem on symmetry operations", *Journal of Mathematical Physics* 5, 862–868 (1964) — the unitary or anti-unitary alternative behind section 4. doi:10.1063/1.1704188

**Previous:** `notebooks/01_QuantumFoundations/06_gates_as_unitary_evolution.ipynb` · **Next:** `notebooks/01_QuantumFoundations/08_pauli_gates.ipynb`